# Agent: Trigger Agent (Flood Event Simulator)

**Purpose:** Simulate flood trigger events on a predictable cycle.

**Cycle:** 30 seconds total
- 25 seconds: LOW severity warning event
- 15 seconds: HIGH severity flood event  
- 5 seconds: Recovery phase

**Publishes to:** `city/flood/trigger` topic as JSON `TriggerEvent` messages

In [ ]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import TriggerEvent

# Load configuration
try:
    cfg = load_config()
    print(f"MQTT Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Base Topic: {cfg.mqtt.base_topic}")
    if cfg.flood:
        print(f"Flood Config: {cfg.flood}")
except Exception as e:
    print(f"Error loading config: {e}")
    cfg = None

MQTT Broker: 127.0.0.1:1883
Base Topic: simulated-city
Flood Config: FloodConfig(trigger_interval_s=10.0, observer_interval_s=5.0, control_threshold=2.0, response_timeout_s=30.0, map_zoom=12, map_center=(40.0, -75.0))


In [ ]:
if cfg:
    mqtt = MqttConnector(cfg.mqtt, client_id_suffix='trigger')
    mqtt.connect()
    if mqtt.wait_for_connection(timeout=5):
        print("✓ Connected to MQTT broker")
    else:
        print("✗ Failed to connect to MQTT broker")
        mqtt = None

✓ Connected to MQTT broker


In [ ]:
def publish_trigger_event(severity: str, event_type: str = "rain"):
    """Publish a trigger event to MQTT."""
    if not mqtt:
        return False
    
    trigger = TriggerEvent(
        event=event_type,
        severity=severity,
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(cfg.mqtt, "trigger")
    payload = json.dumps(trigger.to_json())
    
    mqtt.client.publish(topic, payload, qos=1)
    timestamp = datetime.fromisoformat(trigger.timestamp).strftime("%H:%M:%S")
    print(f"[{timestamp}] Published: {severity.upper()} {event_type} event")
    return True


def run_flood_cycle():
    """Run one complete flood cycle: 25s warning + 15s flood + 5s recovery."""
    
    # Phase 1: Warning (25 seconds of LOW severity)
    warning_duration = 25
    publish_trigger_event("low", "rain")
    for remaining in range(warning_duration - 1, -1, -1):
        if remaining % 5 == 0 and remaining > 0:
            print(f"  Warning phase: {remaining}s remaining")
        time.sleep(1)
    
    # Phase 2: Flood (15 seconds of HIGH severity)
    flood_duration = 15
    publish_trigger_event("high", "rain")
    for remaining in range(flood_duration - 1, -1, -1):
        if remaining % 5 == 0 and remaining > 0:
            print(f"  Flood phase: {remaining}s remaining")
        time.sleep(1)
    
    # Phase 3: Recovery (5 seconds)
    recovery_duration = 5
    print(f"  Recovery phase: {recovery_duration}s...")
    time.sleep(recovery_duration)
    
    print("✓ Cycle complete")

In [9]:
print("Starting trigger simulation...")
print("Cycle: 25s warning → 15s flood → 5s recovery (45s total)")
print("\nPress Ctrl+C to stop\n")

cycle_count = 0
try:
    while True:
        cycle_count += 1
        print(f"\n--- Cycle {cycle_count} ---")
        run_flood_cycle()
except KeyboardInterrupt:
    print("\n\n✓ Trigger agent stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()

Starting trigger simulation...
Cycle: 25s warning → 15s flood → 5s recovery (45s total)

Press Ctrl+C to stop


--- Cycle 1 ---
[13:46:50] Published: LOW rain event
  Warning phase: 20s remaining
  Warning phase: 15s remaining
  Warning phase: 10s remaining
  Warning phase: 5s remaining
[13:47:15] Published: HIGH rain event
  Flood phase: 10s remaining
  Flood phase: 5s remaining
  Recovery phase: 5s...
✓ Cycle complete

--- Cycle 2 ---
[13:47:35] Published: LOW rain event
  Warning phase: 20s remaining
  Warning phase: 15s remaining
  Warning phase: 10s remaining
  Warning phase: 5s remaining
[13:48:00] Published: HIGH rain event
  Flood phase: 10s remaining
  Flood phase: 5s remaining
  Recovery phase: 5s...
✓ Cycle complete

--- Cycle 3 ---
[13:48:20] Published: LOW rain event
  Warning phase: 20s remaining
  Warning phase: 15s remaining
  Warning phase: 10s remaining
  Warning phase: 5s remaining
[13:48:45] Published: HIGH rain event
  Flood phase: 10s remaining
  Flood phase: 5s r

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...




✓ Trigger agent stopped by user


In [ ]:
# Connect MQTT client
connector = MqttConnector(mqtt_cfg, client_id_suffix="trigger")
connector.connect()

# Wait for connection
if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

✓ Connected to MQTT broker


## Connect to MQTT Broker

In [ ]:
import time
from datetime import datetime, timezone
import json

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent

# Load configuration from config.yaml
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

print(f"MQTT Broker: {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Base Topic: {mqtt_cfg.base_topic}")
print(f"Flood Config: {flood_cfg}")

MQTT Broker: 127.0.0.1:1883
Base Topic: simulated-city
Flood Config: FloodConfig(trigger_interval_s=10.0, observer_interval_s=5.0, control_threshold=2.0, response_timeout_s=30.0, map_zoom=12, map_center=(40.0, -75.0))


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


## Setup: Load Configuration and Connect to MQTT

# Agent: Trigger (Flood Simulator)

This notebook simulates a flood trigger event with a predictable cycle:
- Every 30 seconds, a flood event occurs
- Warning issued 25 seconds in advance
- Flood duration: 15 seconds
- Water level rises to ~6-7 meters during flood, then recovers

This mimics a coastal flood scenario where early warning can be provided to evacuate.

# Agent Trigger
This notebook simulates a flood trigger source. It will publish `TriggerEvent` messages to MQTT.
Imports and logic will use `simulated_city` helpers and configuration.